# Frequency probing: KernelSynth and Light TSMixup backgrounds

## tl;dr

This notebook repeats the pure-tone probing protocol with a known added tone
on generated backgrounds. Each of 100 backgrounds per family creates its
own labelled sweep and therefore its own SV for every checkpoint, stage and
task. The main export counts how many of those 100 background datasets have
SV > 0 at the output head, in two panels for KernelSynth and Light TSMixup.
Results are computed below before they are summarised.

## Context & Methods

Frozen Chronos-Bolt-Tiny checkpoints are probed immediately after decoder blocks
0–3 and after the output head. The seven hierarchical binary frequency tasks
are ordered by decision threshold: LL, L, LH, Mid, HL, H and HH. We retain their
existing bands and boundaries.

For each task, unique frequencies are assigned to five class-stratified
prequential blocks. All phases at a frequency remain in the same block. The
first block is charged uniformly at one bit per label; every later block is
encoded by a linear probe trained only on preceding blocks. The scaler and
PCA (up to 20 components) are fitted inside that preceding-data fit, followed
by logistic regression. A fixed seed of 42 governs splits, probe randomness
and the shuffled-label control.

The reported space saving is
\[
SV = 1 - \frac{L_{\mathrm{MDL}}}{n}.
\]
Positive SV means label compression, zero means no saving, and negative values
are retained. The initial uniform block sets a design-dependent upper bound
near 0.8. SV measures the readability of frequency labels from these
representations; it does not measure compression of waveform samples.

### Key assumptions

- Signals are sampled at 512 Hz. The full grid is 2–250 Hz in 1-Hz steps,
  with ten explicitly distinct phases per frequency.
- Context length is the largest \(L\le512\) satisfying \(L\bmod P=0\) and
  \((L-P)\bmod S=0\), avoiding masked patch padding and uncovered samples.
- The published checkpoint and the fifteen retrained geometries use their
  existing, identified weights; no model is retrained here.
- The grouped five-block estimator is the repository's adapted protocol.
  The figure layout follows [Pagani et al., Figure 3](https://arxiv.org/html/2605.06361v1).
- This notebook supplies exports for later report integration. The solar
  analysis and the existing Bayesian results have separate inputs and scope.

### Set up the experiment

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

REPO = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "notebooks" / "support_scripts" / "probing_lib.py").is_file()),
    None,
)
if REPO is None:
    raise RuntimeError("Open this notebook from the patchAliasing repository.")
sys.path.insert(0, str(REPO / "notebooks" / "support_scripts"))

from probing_protocol import ProbeConfig, context_length, make_design
from probing_lib import MODELS

MODE = os.environ.get("PATCHALIASING_PROBING_MODE", "full").strip().lower()
if MODE not in {"full", "smoke"}:
    raise ValueError("PATCHALIASING_PROBING_MODE must be 'full' or 'smoke'.")
cfg = ProbeConfig.smoke() if MODE == "smoke" else ProbeConfig()
models = tuple(model for model in MODELS if model.tag == "retrained-p16-s16") if MODE == "smoke" else MODELS
ROOT = REPO / "notebooks" / "_run" / "probing"
DEVICE = "cpu"
TASK_ORDER = ["LL", "L", "LH", "Mid", "HL", "H", "HH"]
STAGE_ORDER = ["dec_0", "dec_1", "dec_2", "dec_3", "output_head"]

print(f"Mode: {MODE}; device: {DEVICE}; checkpoints: {len(models)}")
if MODE == "smoke":
    display(Markdown("**SMOKE CHECK — reduced design; these results are not report estimates.**"))

## Data

The Appendix G recipes generate 100 independent backgrounds per family.
Light TSMixup uses 1–4 components, with frequencies separated by at least
10 Hz; KernelSynth uses at most five kernels. Each canonical 544-sample draw
is generated once and normalised to unit variance, then shared across all
checkpoints and cropped to their exact context lengths.

Every background receives the same frequency/phase sweep as the pure-tone
notebook. The injected sinusoid has amplitude 1.5 times the canonical
background standard deviation; the sum is not renormalised. Its known
frequency supplies the label, including when the background itself contains
nearby energy.

One background therefore contributes a *dataset* of frequency/phase
variants, rather than a single binary compression outcome for one waveform.
The full result contains 16 × 2 × 100 × 5 × 7 = 112,000 task/stage rows.

In [ ]:
design = make_design(cfg)
coverage = pd.DataFrame([
    {"model": model.tag, "P": model.P, "S": model.S,
     "context_samples": context_length(model.P, model.S)}
    for model in models
])
display(coverage)
print(f"{design['frequency'].nunique()} frequencies × {cfg.n_phases} phases = {len(design)} contexts per sweep")
display(design.head(6))

print(f"Backgrounds per family: {cfg.n_backgrounds}; families: KernelSynth and Light TSMixup")
print(f"Expected task/stage rows: {len(models) * 2 * cfg.n_backgrounds * 5 * 7:,}")

## Results

### Run or resume the background sweeps

The runner saves atomic background/model shards and validates all expected
rows before exporting counts. A failed draw or fit remains an error; it is
never dropped from the denominator. Repeating the same command resumes
matching completed work. Full mode requires exactly 100 valid background
results per family, checkpoint, stage and task.

In [ ]:
from probing_lib import run_complex

results, run_dir = run_complex(ROOT, cfg=cfg, models=models, device=DEVICE)
print(f"Validated rows: {len(results):,}")
print(f"Run directory: {run_dir}")

### Compression counts for Section 5.2

Rows follow the geometry comparison of Table 5. Columns now identify the
seven hierarchical tasks: a cell is the number of background datasets with
positive output-head SV, out of 100 in the full design. KernelSynth and
Light TSMixup are shown separately. The old forecast-recovery
Lock/Net/Control statistic is retained in its original notebook.

In [ ]:
head = results.loc[results["stage"].eq("output_head")].copy()
keys = ["model", "P", "S", "official", "family", "task"]
if head.duplicated(keys + ["background_id"]).any():
    raise ValueError("Duplicate background outcomes cannot enter the count table.")
expected_ids = set(range(cfg.n_backgrounds))
if not head.groupby(keys)["background_id"].agg(lambda values: set(values) == expected_ids).all():
    raise ValueError("A count cell is missing one or more required backgrounds.")
head["compresses"] = head["SV"] > 0
counts = head.groupby(keys, as_index=False).agg(
    compresses=("compresses", "sum"), n=("background_id", "size")
)
model_order = [model.tag for model in models]
for family, label in (("tsmixup", "Light TSMixup"), ("kernelsynth", "KernelSynth")):
    display(Markdown(f"**{label}: count with SV > 0 / {cfg.n_backgrounds} backgrounds ({MODE})**"))
    panel = counts.loc[counts["family"].eq(family)].pivot(index="model", columns="task", values="compresses")
    display(panel.reindex(index=model_order, columns=TASK_ORDER).astype(int))

### Continuous SV distributions

A positive count does not distinguish a small saving from a large one.
The complete CSV and distribution summaries retain all five stages and
negative values. The following reference-checkpoint table reports the
median and interquartile range across backgrounds, together with the
shuffled-control median. These are descriptive spreads, not confidence
intervals.

In [ ]:
reference = results.loc[results["model"].eq("retrained-p16-s16")]
distribution = reference.groupby(["family", "stage", "task"], sort=False).agg(
    median_sv=("SV", "median"),
    q25_sv=("SV", lambda values: values.quantile(.25)),
    q75_sv=("SV", lambda values: values.quantile(.75)),
    median_shuffled_sv=("SV_shuffled", "median"),
)
display(distribution.round(3))

In [ ]:
prefix = "smoke_" if MODE == "smoke" else ""
figure_paths = sorted(run_dir.rglob(f"{prefix}sv_distribution_p16_s16.png"))
if len(figure_paths) != 1:
    raise RuntimeError(f"Expected one reference distribution figure, found {len(figure_paths)}.")
display(Image(filename=str(figure_paths[0]), width=1100))

### Export inventory

In [ ]:
exported = sorted(path for path in run_dir.rglob("*") if path.is_file() and path.suffix in {".png", ".pdf", ".tex", ".txt"})
print(f"{len(exported)} table/figure/caption export files; raw outcomes and distribution summaries are saved as CSV.")
display(pd.DataFrame({"export": [str(path.relative_to(run_dir)) for path in exported]}).head(18))

## Takeaways

Read the integer counts together with the continuous SV distributions and
the shuffled control. A higher count means the frequency task was
compressible on more of the fixed background datasets under this protocol.
It does not by itself establish recovery in the forecast or a loss at
candidate lock sites. The solar series is not used in this experiment.

In [ ]:
if MODE == "smoke":
    display(Markdown("**Smoke validation only:** the reduced-background counts cannot be used as the report's 100-background table."))
else:
    if cfg.n_backgrounds != 100 or not counts["n"].eq(100).all():
        raise ValueError("The report table requires 100 valid outcomes in every cell.")
    display(Markdown("Every displayed full-design count has 100 valid background outcomes. The LaTeX/CSV exports are ready for later Section 5.2 / Appendix G integration; the report source has not been changed."))